# Cellpose su Green (FNC v2) — zero-shot e fine-tuning (Kaggle)

Prima di eseguire:
1. **Settings → Accelerator → GPU T4 x2** e **Settings → Internet → On** (serve per installare Cellpose e scaricare i pesi;
   richiede l'account Kaggle verificato con il telefono).
2. **Add Input**: i due dataset privati `cellseg-code` (serve lo split) e `fnc-green-data` (immagini + maschere pulite).

Uso:
- **Prova breve** (sessione interattiva): nella cella 3 metti `PROBE = True`, poi **Run All**. Lavora su 2 immagini per split
  e fa 2 epoche di fine-tuning su 8 immagini: controlla memoria, tempi e che non ci siano errori. Poi ferma la sessione.
- **Esecuzione completa**: rimetti `PROBE = False` e fai **Save Version → Save & Run All (Commit)**.
- Se la cella 1 dice che non riesce a installare Cellpose con quelle versioni di numpy/scipy, copia il messaggio.
  Se una sessione ha già dato l'errore `_blas_supports_fpe`: **ferma la sessione** e riaprila (l'ambiente riparte pulito).

Uscita: `/kaggle/working/results_cellpose.zip` (solo maschere e configurazione, niente pesi) → scaricala in
`kaggle/results_cellpose.zip` del progetto. Le maschere di **test** si calcolano ora ma si valutano solo il giorno del test.

In [ ]:
# 1) Install Cellpose WITHOUT touching numpy and scipy: the kernel has already loaded them, and changing their
#    version in the middle of a session breaks every later import ("... has no attribute '_blas_supports_fpe'")
import numpy
import scipy
NUMPY, SCIPY = numpy.__version__, scipy.__version__
print("keeping numpy", NUMPY, "and scipy", SCIPY)
!pip install -q cellpose "numpy=={NUMPY}" "scipy=={SCIPY}"
from importlib.metadata import version
CP_VERSION = version("cellpose")
MAJOR = int(CP_VERSION.split(".")[0])
print("cellpose", CP_VERSION)

In [ ]:
# 2) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "no GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3) Data: images, cleaned masks and the train/val split used by our models
import json
import time
from pathlib import Path

import numpy as np
import skimage.io as io
from skimage import measure

PROBE = False        # True = short check (2 images per split, 2 epochs of fine-tuning on 8 images)

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/runs/cellpose" + ("_probe" if PROBE else ""))
OUT.mkdir(parents=True, exist_ok=True)

def find_one(pattern):
    '''First path under /kaggle/input matching the pattern (Kaggle may nest the dataset folders).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (did you add both datasets with 'Add Input'?)"
    return hits[0]

split = json.load(open(find_one("splits/green_split.json")))
TRAINVAL, TEST = find_one("green/trainval/images"), find_one("green/test/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
SPLITS = {"train": (TRAINVAL, split["train"]), "val": (TRAINVAL, split["val"]),
          "test": (TEST, sorted(p.name for p in TEST.glob("*.png")))}
assert [len(names) for _, names in SPLITS.values()] == [169, 43, 70], "expected 169 / 43 / 70 images"
if PROBE:
    SPLITS = {s: (folder, names[:2]) for s, (folder, names) in SPLITS.items()}
for s, (folder, names) in SPLITS.items():
    print(f"{s}: {len(names)} images from {folder}")

def signal(folder, name):
    '''The G channel (the c-FOS signal) as a 2-D uint8 image.'''
    return io.imread(folder / name)[..., 1]

In [ ]:
# 4) Helpers: segment a split, save the instance masks
DIAMETER = 30        # px: mean equivalent diameter of the train cells (30.6 px); Cellpose thresholds left at their defaults

def segment(model, split_name):
    '''Instance masks (uint16, 0 = background) of every image of a split, and seconds per image.'''
    folder, names = SPLITS[split_name]
    masks, start = [], time.time()
    for name in names:
        masks.append(model.eval(signal(folder, name), diameter=DIAMETER)[0].astype(np.uint16))
    return np.stack(masks), (time.time() - start) / len(names)

def save(masks, split_name, prefix):
    np.savez_compressed(OUT / f"{prefix}_{split_name}.npz", masks=masks, names=np.array(SPLITS[split_name][1]))

timing = {}

## Zero-shot (nessun training sulle nostre etichette)
Modello predefinito della versione installata (Cellpose-SAM dalla versione 4); con versioni precedenti il modello `nuclei`
(il c-FOS colora i nuclei). Controllo: su due immagini di train, numero di oggetti accanto al numero di cellule vere.

In [ ]:
# 5) Zero-shot model + check on two TRAIN images
from cellpose import models, train

def base_model():
    return models.CellposeModel(gpu=True) if MAJOR >= 4 else models.CellposeModel(gpu=True, model_type="nuclei")

zero_shot = base_model()
folder, names = SPLITS["train"]
for name in names[:2]:
    start = time.time()
    m = zero_shot.eval(signal(folder, name), diameter=DIAMETER)[0]
    n_true = measure.label(io.imread(MASKS / name) > 0, connectivity=1).max()
    print(f"{name}: shape {m.shape}, {m.max()} objects vs {n_true} true cells, {time.time() - start:.1f} s")

In [ ]:
# 6) Zero-shot on train, val and test
for s in SPLITS:
    masks, sec = segment(zero_shot, s)
    save(masks, s, "cellpose")
    timing[f"zero_shot_{s}_s_per_image"] = sec
    print(f"zero-shot {s}: {len(masks)} images, {sec:.1f} s/image")
del zero_shot
torch.cuda.empty_cache()

# safety copy: if the long fine-tuning stops, the zero-shot results are not lost
import shutil
shutil.make_archive("/kaggle/working/results_cellpose_zeroshot" + ("_probe" if PROBE else ""), "zip",
                    root_dir="/kaggle/working", base_dir=str(OUT.relative_to("/kaggle/working")))

## Fine-tuning sulle nostre etichette (solo le 169 immagini di TRAIN: la val non entra mai)
Stesso modello dello zero-shot, impostazioni predefinite di `train_seg` della versione installata (registrate nel file di
configurazione), numero di epoche **fissato prima** (nessuna epoca scelta guardando la val). Etichette di istanza dalle
maschere pulite (connectivity 1: le cellule annotate non si toccano mai). Unica eccezione ai valori predefiniti:
`min_train_masks=0`. Di default Cellpose scarterebbe le immagini con meno di 5 cellule (29 su 169), cioè proprio quelle
in cui gli annotatori hanno lasciato fuori le cellule deboli: così usa le stesse 169 immagini di v2.

In [ ]:
# 7) Fine-tuning on train (all 169 images, as v2)
import inspect

N_EPOCHS = 2 if PROBE else 100
names = split["train"]
cells = {n: measure.label(io.imread(MASKS / n) > 0, connectivity=1) for n in names}
if PROBE:   # 6 images + the train images without cells: checks that empty images do not break the training
    names = names[:6] + [n for n in names if cells[n].max() == 0]
train_data = [signal(TRAINVAL, n) for n in names]
train_labels = [cells[n].astype(np.int32) for n in names]
print(f"fine-tuning images: {len(names)}, of which without cells: {sum(cells[n].max() == 0 for n in names)}")

MODEL_DIR = Path("/kaggle/working/cellpose_model")   # weights outside the results zip
MODEL_DIR.mkdir(exist_ok=True)                       # train_seg only creates the "models" subfolder
tuned = base_model()
start = time.time()
out = train.train_seg(tuned.net, train_data=train_data, train_labels=train_labels, n_epochs=N_EPOCHS,
                      min_train_masks=0, save_path=str(MODEL_DIR), model_name="cellpose_ft_green")
model_path = out[0] if isinstance(out, tuple) else out
timing["fine_tuning_minutes"] = (time.time() - start) / 60
losses = np.asarray(out[1], dtype=float) if isinstance(out, tuple) and len(out) > 1 else np.array([])
defaults = {k: repr(v.default) for k, v in inspect.signature(train.train_seg).parameters.items()
            if v.default is not inspect.Parameter.empty}
print(f"fine-tuning: {len(train_data)} images, {N_EPOCHS} epochs, {timing['fine_tuning_minutes']:.1f} min → {model_path}")
# the "Mean of empty slice" warnings come from the images without cells (no diameter): harmless if rescale is False
print(f"rescale by cell diameter: {defaults.get('rescale')} | training loss: first {losses[:1]}, last {losses[-3:]}")
assert np.isfinite(losses).all(), "NaN/inf in the training loss"
del tuned
torch.cuda.empty_cache()

In [ ]:
# 8) Fine-tuned model on val and test (not on train: it has seen those images, so they would not be a fair comparison)
tuned = models.CellposeModel(gpu=True, pretrained_model=str(model_path))
for s in ["val", "test"]:
    masks, sec = segment(tuned, s)
    save(masks, s, "cellpose_ft")
    timing[f"fine_tuned_{s}_s_per_image"] = sec
    print(f"fine-tuned {s}: {len(masks)} images, {sec:.1f} s/image")

In [ ]:
# 9) Configuration + one zip with masks and configuration only
import shutil

config = {"cellpose_version": CP_VERSION, "model": "default of the version" if MAJOR >= 4 else "nuclei",
          "input": "G channel, 2-D uint8", "diameter": DIAMETER, "eval": "Cellpose default thresholds",
          "fine_tuning": {"train_images": len(train_data), "n_epochs": N_EPOCHS, "min_train_masks": 0,
                          "train_seg_defaults": defaults},
          "timing": timing, "probe": PROBE}
with open(OUT / "cellpose_config.json", "w") as f:
    json.dump(config, f, indent=2)
shutil.make_archive("/kaggle/working/results_cellpose" + ("_probe" if PROBE else ""), "zip",
                    root_dir="/kaggle/working", base_dir=str(OUT.relative_to("/kaggle/working")))
print(json.dumps(config["timing"], indent=2))
print(sorted(p.name for p in OUT.iterdir()))